# `helpers/news.py` — Playground

| Function | Notes |
|---|---|
| `classify_source(source_name, url=None)` | URL fallback when source is generic (benzinga) |
| `fetch_news(...)` | Alpaca → Finnhub → NewsAPI; optional `include_summary` |
| `format_news_for_prompt(news_items)` | Gate 2/3 prompt block |

In [1]:
import sys
import pathlib

helpers_dir = pathlib.Path('backend/02_intelligence/helpers/fetchers').resolve()
if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from news import classify_source, fetch_news, format_news_for_prompt

## `classify_source()`

In [2]:
for source, url in [
    ('Reuters', None),
    ('benzinga', 'https://www.reuters.com/markets/'),
    ('Yahoo', None),
    ('Unknown Publisher', None),
]:
    print(f'{source!r:20} → {classify_source(source, url)}')

'Reuters'            → HIGH
'benzinga'           → HIGH
'Yahoo'              → MEDIUM
'Unknown Publisher'  → LOW


## `fetch_news(ticker)` — happy path

Each dict: `headline`, `source`, `reliability`, `url`, `published_at`, `summary`

In [3]:
news = fetch_news('NVDA')
if news is None:
    print('FETCH FAILED — check .env keys')
elif not news:
    print('No articles in window')
else:
    print(f'{len(news)} articles')
    for item in news:
        print(item)

5 articles
{'headline': "Jim Cramer Says Dell's Blowout Quarter Could Mark A Turning Point For AI Stocks Like Nvidia And Intel: 'I Wonder If...'", 'source': 'benzinga', 'reliability': 'HIGH', 'url': 'https://www.benzinga.com/markets/tech/26/05/52892441/jim-cramer-says-dells-blowout-quarter-could-mark-a-turning-point-for-ai-stocks-like-nvidia-and-intel-i-wonder-if', 'published_at': '2026-05-30T03:58:00+00:00', 'summary': 'Cramer said Dell&#39;s blowout earnings could mark a turning point for AI stocks as investors turn to Nvidia&#39;s Computex event.'}
{'headline': 'The S&P Hit A Record While 8 Of 11 Sectors Fell', 'source': 'benzinga', 'reliability': 'HIGH', 'url': 'https://www.benzinga.com/Opinion/26/05/52891278/the-sp-hit-a-record-while-8-of-11-sectors-fell', 'published_at': '2026-05-29T21:40:15+00:00', 'summary': 'Dell ripped roughly 30% on $16.1B in quarterly AI server sales. Eight of eleven sectors still finished May in the red. Inside: oil cracks on a 60-day Iran ceasefire, and A

## `include_summary=False`

In [4]:
with_summary = fetch_news('NVDA', max_results=2, include_summary=True)
without_summary = fetch_news('NVDA', max_results=2, include_summary=False)
if with_summary and without_summary:
    print('with summary:   ', repr(with_summary[0].get('summary', '')[:80]))
    print('without summary:', repr(without_summary[0].get('summary', '')))

with summary:    'Cramer said Dell&#39;s blowout earnings could mark a turning point for AI stocks'
without summary: ''


## `format_news_for_prompt()`

In [5]:
if news:
    print(format_news_for_prompt(news))
else:
    print('Run happy path cell first')

Jim Cramer Says Dell's Blowout Quarter Could Mark A Turning Point For AI Stocks Like Nvidia And Intel: 'I Wonder If...' [Source: benzinga] [Reliability: HIGH] — Cramer said Dell&#39;s blowout earnings could mark a turning point for AI stocks as investors turn to Nvidia&#39;s Computex event.
The S&P Hit A Record While 8 Of 11 Sectors Fell [Source: benzinga] [Reliability: HIGH] — Dell ripped roughly 30% on $16.1B in quarterly AI server sales. Eight of eleven sectors still finished May in the red. Inside: oil cracks on a 60-day Iran ceasefire, and Anthropic lines up a $900B raise.
Tuttle Capital Launches Photonics ETF To Target AI Infrastructure's Next Big Bottleneck [Source: benzinga] [Reliability: HIGH] — The FOTO ETF targets photonics stocks poised to benefit from the AI infrastructure boom and hyperscaler data center spending.
Buying On Iran Deal; Dell Earnings Support Semi Rally; Rocket Explosion Is Bad News For Space Mania [Source: benzinga] [Reliability: HIGH] — Stock Market Manias

## Failure path

In [ ]:
bad = fetch_news('ZZZZINVALID')
print(f'ZZZZINVALID → {bad!r}')

## Free-play